In [1]:
from copy import deepcopy

import matplotlib.pyplot as plt
%config InlineBackend.figure_format='retina'

from acm.utils.modules import get_class_from_module
from sunbird.inference.samples.tables import get_subtable_dict, get_table
from utils import get_chains, print_std_improvements, plot_model_vs_truth
from secondgen_bgs import get_secondgen_data # FIXME: Will be moved to bgs_mock.py

chain_dir_Mr20 = '/pscratch/sd/s/sbouchar/acm/bgs-20/chains/'
chain_dir_Mr21 = '/pscratch/sd/s/sbouchar/acm/bgs-21.35/chains/'

cosmo_params = ['omega_cdm', 'omega_b', 'sigma8_m', 'n_s', 'nrun', 'N_ur', 'w0_fld', 'wa_fld']
hod_params = ['logM_cut', 'logM_1', 'alpha', 'kappa', 'sigma', 'alpha_c', 'alpha_s', 's', 'A_cen', 'A_sat', 'B_cen', 'B_sat']

# default values
colors = ["#5ac3be", "#e770a2", "#4165c0", "#696969", "#f79a1e", "#ba7dcd"]

label_dict = {
    'tpcf': '2PCF', 
    'ds_xiqq+ds_xiqg': 'DS',
    'ds_xiqg+ds_xiqq': 'DS',
    'tpcf+ds_xiqq+ds_xiqg': '2PCF + DS',
    'tpcf+ds_xiqg+ds_xiqq': '2PCF + DS',
    'tpcf_s_1.00-150.00': '2PCF (1-150 Mpc/h)',
    'tpcf_s_30.00-150.00': '2PCF (30-150 Mpc/h)',
    'tpcf_s_1.00-150.00+ds_xiqq_s_1.00-30.00+ds_xiqg_s_1.00-30.00': '2PCF (1-150 Mpc/h) + DS (1-30 Mpc/h)',
    'tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00': '2PCF (1-150 Mpc/h) + DS (1-30 Mpc/h)',
}

# For specific uses
colors2 = ["#182a88", "#105203", "#9ca6d8", "#8fde7f"]
label_dict2 = {
    "tpcf_s_1.00-150.00 (Mr20)": "2PCF (Mr < -20)",
    "tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00 (Mr20)": "2PCF + DS (Mr < -20)",
    "tpcf_s_1.00-150.00 (Mr21)": "2PCF (Mr < -21)",
    "tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00 (Mr21)": "2PCF + DS (Mr < -21)",
}

## Validation plots (c000 hod000)

In [ ]:
chains = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_30.00-150.00', 'tpcf_s_1.00-150.00'],
    chain_dir=chain_dir_Mr21,
)

print('Standard deviation improvements for small-scales 2PCF (1-150 Mpc/h):')
print_std_improvements(chains[0], chains[1], cosmo_params)

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle('w0wa fit to c000 validation mock (Mr<-21.35)');

In [ ]:
chains = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_1.00-150.00', 'tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00'],
    chain_dir=chain_dir_Mr21,
)

print('Standard deviation improvements with DS (1-30 Mpc/h) added to 2PCF (1-150 Mpc/h):')
print_std_improvements(chains[0], chains[1], cosmo_params)

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle('w0wa fit to c000 validation mock (Mr<-21.35)');

In [ ]:
chains_Mr21 = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_1.00-150.00', 'tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00'],
    chain_dir=chain_dir_Mr21,
)

for chain in chains_Mr21:
    chain.data['label'] += ' (Mr21)'

chains_Mr20 = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_1.00-150.00', 'tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00'],
    chain_dir=chain_dir_Mr20,
)

for chain in chains_Mr20:
    chain.data['label'] += ' (Mr20)'
    
print('Standard deviation improvements with Mr<-20 (2PCF):')
print_std_improvements(chains_Mr21[0], chains_Mr20[0], cosmo_params)

print('Standard deviation improvements with Mr<-20 (2PCF+DS):')
print_std_improvements(chains_Mr21[1], chains_Mr20[1], cosmo_params)

chains = chains_Mr21 + chains_Mr20

markers = chains[0].markers
chains[0].plot_triangle(
    *chains[1:], 
    add_bestfit=True, 
    colors=colors2, 
    filled=True, 
    title_limit=1, 
    label_dict=label_dict2, 
    markers=markers, 
    params=cosmo_params, 
    contour_args=dict(alpha=0.9)
);

fig = plt.gcf()
fig.suptitle('w0wa fit to c000 validation mock ');


chains[0].plot_map(
    *chains[1:], 
    percentile = [16, 84],
    colors=colors2,
    label_dict=label_dict2, 
    markers=markers, 
    params=cosmo_params, 
);

In [ ]:
chains_Mr21 = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_30.00-150.00'],
    chain_dir=chain_dir_Mr21,
)

for chain in chains_Mr21:
    chain.data['label'] += ' (Mr21)'

chains_Mr20 = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_30.00-150.00'],
    chain_dir=chain_dir_Mr20,
)

for chain in chains_Mr20:
    chain.data['label'] += ' (Mr20)'
    
print('Standard deviation improvements with Mr<-20 (2PCF):')
print_std_improvements(chains_Mr21[0], chains_Mr20[0], cosmo_params)

chains = chains_Mr21 + chains_Mr20

markers = chains[0].markers
chains[0].plot_triangle(
    *chains[1:], 
    add_bestfit=True, 
    colors=colors2, 
    filled=True, 
    title_limit=1, 
    label_dict=label_dict2, 
    markers=markers, 
    params=cosmo_params, 
    contour_args=dict(alpha=0.9)
);

fig = plt.gcf()
fig.suptitle('w0wa fit to c000 validation mock ');

In [ ]:
# Total improvement

chain_base = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_30.00-150.00'],
    chain_dir=chain_dir_Mr21,
)[0]

chain_final = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00'],
    chain_dir=chain_dir_Mr20,
)[0]

print('Total standard deviation improvements (Mr<-20, 2PCF 1-150 Mpc/h + DS 1-30 Mpc/h):')
print_std_improvements(chain_base, chain_final, cosmo_params);

In [ ]:
# Comparaison table 

stats = {
    'tpcf_s_30.00-150.00': r'2PCF (30-150) h^{-1} Mpc', 
    'tpcf_s_1.00-150.00': r'2PCF (1-150) h^{-1} Mpc',
    'ds_xiqg_s_1.00-150.00+ds_xiqq_s_1.00-150.00': r'DS (1-150) h^{-1} Mpc',
    'tpcf_s_1.00-150.00+ds_xiqg_s_1.00-150.00+ds_xiqq_s_1.00-150.00': r'2PCF + DS (1-150) h^{-1} Mpc',
    'tpcf_s_1.00-150.00+ds_xiqg_s_1.00-30.00+ds_xiqq_s_1.00-30.00': r'2PCF (1-150) + DS (1-30) h^{-1} Mpc',
}

chains_Mr21 = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=list(stats.keys()),
    chain_dir=chain_dir_Mr21,
)

chains_Mr20 = get_chains(
    type_fit='validation/c000_hod000', 
    cosmo_model='base-w0wa-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=list(stats.keys()),
    chain_dir=chain_dir_Mr20,
)

st_21 = get_subtable_dict(*chains_Mr21, params=cosmo_params, low_error_only=False)
st_20 = get_subtable_dict(*chains_Mr20, params=cosmo_params, low_error_only=False)
latex_table = get_table(
    {'BGS Mr<-21.35': st_21, 'BGS Mr<-20': st_20}, 
    header_name='Configuration', 
    label_dict=stats,
)

print(latex_table)

In [ ]:
# Model vs truth
label = chain_final.data['label']
stats = label.split('+')

Mr = -20
paths = dict(
    data_dir= f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/input_data/',
    model_dir= f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/trained_models/',
)

for stat in stats:
    stat_name = stat.split('_s')[0]
    data_obs = get_class_from_module('acm.observables.bgs', stat_name)(paths=paths, select_filters={'cosmo_idx': 0, 'hod_idx': 0})
    fig, ax = plot_model_vs_truth(chain_final, data_obs=data_obs)
    fig.suptitle(rf'w0wa (fixed $\Omega_b$) fit on validation mock - {label_dict[label]}');

## SecondGen plots

### LCDM

In [ ]:
chains = get_chains(
    type_fit='secondgen', 
    cosmo_model='base-fixed-omega_b', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf', 'tpcf+ds_xiqg+ds_xiqq'],
    chain_dir = chain_dir_Mr20, 
)

print('Standard deviation improvements from validation mock to secondgen:')
print_std_improvements(chains[0], chains[1], cosmo_params)

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle(r'LCDM (fixed $\Omega_b$) fit on SecondGen data (Mr<-20)');

### w0wa

In [ ]:
chains = get_chains(
    type_fit='secondgen', 
    cosmo_model='base-w0wa', 
    hod_model='base-AB-CB-VB-s',
    stats=['tpcf', 'tpcf+ds_xiqg+ds_xiqq'],
    chain_dir = chain_dir_Mr20, 
)

print('Standard deviation improvements from validation mock to secondgen:')
print_std_improvements(chains[0], chains[1], cosmo_params)

markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params);

fig = plt.gcf()
fig.suptitle('w0wa fit on SecondGen data');

### Model prediction vs truth for SecondGen

In [ ]:
chains = get_chains(type_fit='secondgen', cosmo_model='base-fixed-omega_b', hod_model='base-AB-CB-VB-s', chain_dir='/pscratch/sd/s/sbouchar/acm/bgs-20/chains/')

Mr = -20
paths = dict(
    data_dir= f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/input_data/',
    model_dir= f'/pscratch/sd/s/sbouchar/acm/bgs{Mr}/trained_models/',
)

for chain in chains:
    label = chain.data['label']
    stats = label.split('+')
    for stat in stats:
        stat_name = stat.split('_s')[0]
        model_obs = get_class_from_module('acm.observables.bgs', stat_name)(paths=paths)
        data_obs = deepcopy(model_obs)
        data_obs._dataset = get_secondgen_data(model_obs, return_obs=True)[0]._dataset # Contains observed data + covariance 
        
        fig, ax = plot_model_vs_truth(chain, data_obs=data_obs, model_obs=model_obs, volume_factor=1.0)
        fig.suptitle(rf'LCDM (fixed $\Omega_b$) fit on SecondGen data - {label}');

## Old tests

### First plot testing the secondgen inference 

In [ ]:
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt

from sunbird.inference.samples import Chain
from acm.observables.bgs import tpcf
from secondgen_bgs import get_secondgen_data # FIXME: Will be moved to bgs_mock.py

obs = tpcf(select_filters=dict(ells=0), numpy_output=True)
s = obs.s

sg_tpcf = get_secondgen_data(obs, return_obs=True)
truth = sg_tpcf.y
sg_err = np.sqrt(np.diag(sg_tpcf.get_covariance_matrix(volume_factor=1.0)))

pred = obs.get_model_prediction(sg_tpcf.x)

model_err = obs.get_emulator_error()
mock_err = np.sqrt(np.diag(obs.get_covariance_matrix()))
pred_err = np.sqrt(model_err**2 + mock_err**2)

chain = Chain.load('/pscratch/sd/s/sbouchar/acm/bgs-20/chains/secondgen/cosmo-base_hod-base-VB-AB-CB-s/tpcf.npy')

pred_fit = obs.get_model_prediction(chain.bestfit)

plt.errorbar(s, truth*s**2, yerr=sg_err*s**2, fmt='o', ms=3, label='SecondGen BGS')

plt.plot(s, pred*s**2, label='ACM Truth Prediction')
plt.fill_between(s, (pred - pred_err)*s**2, (pred + pred_err)*s**2, color='C1', alpha=0.3)

plt.plot(s, pred_fit*s**2, label='ACM Best-fit Prediction', linestyle='--')
plt.fill_between(s, (pred_fit - pred_err)*s**2, (pred_fit + pred_err)*s**2, color='C2', alpha=0.3)

plt.xlabel('s [Mpc/h]')
plt.ylabel(r'$s^2 \xi_0(s)$')
plt.title('BGS Monopole Correlation Function')
plt.legend();

### Plot examples

In [ ]:
# SecondGen LCDM w/ Full HOD
type_fit = 'secondgen'
cosmo_model = 'base'
hod_model = 'base-VB-AB-CB-s'

# Triange plot
chains = get_chains(type_fit, cosmo_model, hod_model)
markers = chains[0].markers
chains[0].plot_triangle(*chains[1:], add_bestfit=True, colors=colors, filled=True, title_limit=1, label_dict=label_dict, markers=markers, params=cosmo_params)

# MAP
chains[0].plot_map(*chains[1:], colors=colors, label_dict=label_dict, markers=markers, params=cosmo_params);

# Model vs Truth plot
model_obs = get_class_from_module('acm.observables.bgs', 'ds_xiqq')()
data_obs = deepcopy(model_obs)
data_obs._dataset = get_secondgen_data(model_obs, return_obs=True)[0]._dataset # Contains observed data + covariance 
fig, ax = plot_model_vs_truth(chains[1], data_obs=data_obs, model_obs=model_obs, volume_factor=1.0)